# 23 - SQLite Vertiefung: Fortgeschrittene Abfragen und Datenmanipulation

## Lernziele

Nach Abschluss dieses Notebooks können Sie:
- Daten mit WHERE, ORDER BY und komplexen Bedingungen gezielt filtern und sortieren
- Tabellen mit INNER JOIN und LEFT JOIN verknüpfen und Beziehungen abfragen
- Daten mit UPDATE und DELETE gezielt ändern und löschen
- Aggregatfunktionen (COUNT, SUM, AVG) mit GROUP BY für Auswertungen nutzen

**Kompetenzstufen**: Anwenden, Analysieren

---

## Voraussetzungen

Für dieses Notebook sollten Sie folgende Konzepte beherrschen:
- Grundlagen relationaler Datenbanken: Tabellen, Primär- und Fremdschlüssel (Notebook 21)
- SQLite-Grundlagen: connect(), cursor(), execute(), fetchall(), commit() (Notebook 22)
- CREATE TABLE und INSERT-Befehle (Notebook 22)
- Parametrisierte Abfragen mit ? (Notebook 22)

Falls Sie diese Konzepte noch nicht sicher beherrschen, wiederholen Sie bitte die entsprechenden Notebooks.

---

## Vorbereitung: Die Chinook-Datenbank

In diesem Notebook arbeiten wir mit der **Chinook-Datenbank**, einer Beispieldatenbank, die einen digitalen Musikshop simuliert. Sie enthält Informationen über Künstler, Alben, Tracks, Kunden und Rechnungen.

Die wichtigsten Tabellen für dieses Notebook sind:
- `artists`: Künstler (ArtistId, Name)
- `albums`: Alben (AlbumId, Title, ArtistId)
- `tracks`: Musikstücke (TrackId, Name, AlbumId, GenreId, Composer, Milliseconds, UnitPrice)
- `genres`: Musikgenres (GenreId, Name)
- `customers`: Kunden (CustomerId, FirstName, LastName, Country, City)
- `invoices`: Rechnungen (InvoiceId, CustomerId, InvoiceDate, Total)

In [2]:
# Import des sqlite3-Moduls
import sqlite3

# Hilfsfunktion für Abfragen (aus Notebook 22 bekannt)
def query_db(query, params=(), limit=10):
    """Führt eine SELECT-Abfrage aus und zeigt die Ergebnisse."""
    with sqlite3.connect('chinook.db') as conn:
        cursor = conn.cursor()
        cursor.execute(query, params)
        ergebnisse = cursor.fetchall()
        
        print(f"{len(ergebnisse)} Ergebnisse gefunden.")
        print("-" * 60)
        
        for zeile in ergebnisse[:limit]:
            print(zeile)
        
        if len(ergebnisse) > limit:
            print(f"... und {len(ergebnisse) - limit} weitere Ergebnisse")
        
        return ergebnisse

---

## Kapitel 1: Daten filtern mit WHERE

### Einführung und Motivation

In Notebook 22 haben Sie gelernt, mit `SELECT` alle Daten aus einer Tabelle abzurufen. In der Praxis benötigen Sie jedoch selten alle Daten – stattdessen suchen Sie nach spezifischen Informationen. Die `WHERE`-Klausel ermöglicht es, Ergebnisse auf Datensätze zu beschränken, die bestimmten Bedingungen entsprechen.

Stellen Sie sich vor, Sie verwalten eine Musikdatenbank mit tausenden Tracks. Ohne Filterung müssten Sie alle Einträge manuell durchsuchen. Mit `WHERE` können Sie gezielt nach Rock-Songs, Tracks eines bestimmten Komponisten oder Liedern mit einem Preis unter 1€ suchen.

Die `WHERE`-Klausel ist das wichtigste Werkzeug für präzise Datenbankabfragen und bildet die Grundlage für alle fortgeschrittenen Filteroperationen.

### Syntax der WHERE-Klausel

**Grundsyntax:**
```sql
SELECT spalte1, spalte2, ...
FROM tabelle
WHERE bedingung;
```

**Verfügbare Vergleichsoperatoren:**

| Operator | Bedeutung | Beispiel |
|----------|-----------|----------|
| `=` | Gleich | `WHERE jahr = 2024` |
| `!=` oder `<>` | Ungleich | `WHERE status != 'aktiv'` |
| `<` | Kleiner als | `WHERE preis < 10` |
| `>` | Größer als | `WHERE alter > 18` |
| `<=` | Kleiner oder gleich | `WHERE menge <= 100` |
| `>=` | Größer oder gleich | `WHERE punkte >= 50` |

### Beispiel 1: Einfache Bedingung

Wir suchen alle Tracks, die genau 0.99€ kosten:

In [7]:
query_db("SELECT Name, UnitPrice FROM tracks")

3503 Ergebnisse gefunden.
------------------------------------------------------------
('For Those About To Rock (We Salute You)', 0.99)
('Balls to the Wall', 0.99)
('Fast As a Shark', 0.99)
('Restless and Wild', 0.99)
('Princess of the Dawn', 0.99)
('Put The Finger On You', 0.99)
("Let's Get It Up", 0.99)
('Inject The Venom', 0.99)
('Snowballed', 0.99)
('Evil Walks', 0.99)
... und 3493 weitere Ergebnisse


[('For Those About To Rock (We Salute You)', 0.99),
 ('Balls to the Wall', 0.99),
 ('Fast As a Shark', 0.99),
 ('Restless and Wild', 0.99),
 ('Princess of the Dawn', 0.99),
 ('Put The Finger On You', 0.99),
 ("Let's Get It Up", 0.99),
 ('Inject The Venom', 0.99),
 ('Snowballed', 0.99),
 ('Evil Walks', 0.99),
 ('C.O.D.', 0.99),
 ('Breaking The Rules', 0.99),
 ('Night Of The Long Knives', 0.99),
 ('Spellbound', 0.99),
 ('Go Down', 0.99),
 ('Dog Eat Dog', 0.99),
 ('Let There Be Rock', 0.99),
 ('Bad Boy Boogie', 0.99),
 ('Problem Child', 0.99),
 ('Overdose', 0.99),
 ("Hell Ain't A Bad Place To Be", 0.99),
 ('Whole Lotta Rosie', 0.99),
 ('Walk On Water', 0.99),
 ('Love In An Elevator', 0.99),
 ('Rag Doll', 0.99),
 ('What It Takes', 0.99),
 ('Dude (Looks Like A Lady)', 0.99),
 ("Janie's Got A Gun", 0.99),
 ("Cryin'", 0.99),
 ('Amazing', 0.99),
 ('Blind Man', 0.99),
 ('Deuces Are Wild', 0.99),
 ('The Other Side', 0.99),
 ('Crazy', 0.99),
 ('Eat The Rich', 0.99),
 ('Angel', 0.99),
 ("Livin' On

In [3]:
# Tracks mit Preis von 0.99€
query_db("SELECT Name, UnitPrice FROM tracks WHERE UnitPrice = 0.99")

3290 Ergebnisse gefunden.
------------------------------------------------------------
('For Those About To Rock (We Salute You)', 0.99)
('Balls to the Wall', 0.99)
('Fast As a Shark', 0.99)
('Restless and Wild', 0.99)
('Princess of the Dawn', 0.99)
('Put The Finger On You', 0.99)
("Let's Get It Up", 0.99)
('Inject The Venom', 0.99)
('Snowballed', 0.99)
('Evil Walks', 0.99)
... und 3280 weitere Ergebnisse


[('For Those About To Rock (We Salute You)', 0.99),
 ('Balls to the Wall', 0.99),
 ('Fast As a Shark', 0.99),
 ('Restless and Wild', 0.99),
 ('Princess of the Dawn', 0.99),
 ('Put The Finger On You', 0.99),
 ("Let's Get It Up", 0.99),
 ('Inject The Venom', 0.99),
 ('Snowballed', 0.99),
 ('Evil Walks', 0.99),
 ('C.O.D.', 0.99),
 ('Breaking The Rules', 0.99),
 ('Night Of The Long Knives', 0.99),
 ('Spellbound', 0.99),
 ('Go Down', 0.99),
 ('Dog Eat Dog', 0.99),
 ('Let There Be Rock', 0.99),
 ('Bad Boy Boogie', 0.99),
 ('Problem Child', 0.99),
 ('Overdose', 0.99),
 ("Hell Ain't A Bad Place To Be", 0.99),
 ('Whole Lotta Rosie', 0.99),
 ('Walk On Water', 0.99),
 ('Love In An Elevator', 0.99),
 ('Rag Doll', 0.99),
 ('What It Takes', 0.99),
 ('Dude (Looks Like A Lady)', 0.99),
 ("Janie's Got A Gun", 0.99),
 ("Cryin'", 0.99),
 ('Amazing', 0.99),
 ('Blind Man', 0.99),
 ('Deuces Are Wild', 0.99),
 ('The Other Side', 0.99),
 ('Crazy', 0.99),
 ('Eat The Rich', 0.99),
 ('Angel', 0.99),
 ("Livin' On

In [5]:
query_db("SELECT Name, UnitPrice FROM tracks WHERE UnitPrice != 0.99")

213 Ergebnisse gefunden.
------------------------------------------------------------
('Battlestar Galactica: The Story So Far', 1.99)
('Occupation / Precipice', 1.99)
('Exodus, Pt. 1', 1.99)
('Exodus, Pt. 2', 1.99)
('Collaborators', 1.99)
('Torn', 1.99)
('A Measure of Salvation', 1.99)
('Hero', 1.99)
('Unfinished Business', 1.99)
('The Passage', 1.99)
... und 203 weitere Ergebnisse


[('Battlestar Galactica: The Story So Far', 1.99),
 ('Occupation / Precipice', 1.99),
 ('Exodus, Pt. 1', 1.99),
 ('Exodus, Pt. 2', 1.99),
 ('Collaborators', 1.99),
 ('Torn', 1.99),
 ('A Measure of Salvation', 1.99),
 ('Hero', 1.99),
 ('Unfinished Business', 1.99),
 ('The Passage', 1.99),
 ('The Eye of Jupiter', 1.99),
 ('Rapture', 1.99),
 ('Taking a Break from All Your Worries', 1.99),
 ('The Woman King', 1.99),
 ('A Day In the Life', 1.99),
 ('Dirty Hands', 1.99),
 ('Maelstrom', 1.99),
 ('The Son Also Rises', 1.99),
 ('Crossroads, Pt. 1', 1.99),
 ('Crossroads, Pt. 2', 1.99),
 ('Genesis', 1.99),
 ("Don't Look Back", 1.99),
 ('One Giant Leap', 1.99),
 ('Collision', 1.99),
 ('Hiros', 1.99),
 ('Better Halves', 1.99),
 ('Nothing to Hide', 1.99),
 ('Seven Minutes to Midnight', 1.99),
 ('Homecoming', 1.99),
 ('Six Months Ago', 1.99),
 ('Fallout', 1.99),
 ('The Fix', 1.99),
 ('Distractions', 1.99),
 ('Run!', 1.99),
 ('Unexpected', 1.99),
 ('Company Man', 1.99),
 ('Company Man', 1.99),
 ('Para

Die Abfrage gibt nur Tracks zurück, bei denen der Preis exakt 0.99 beträgt. Alle anderen Datensätze werden herausgefiltert.

### Beispiel 2: Textvergleiche

Bei Textvergleichen müssen Werte in Anführungszeichen gesetzt werden:

In [8]:
# Alle Kunden aus Deutschland
query_db("SELECT FirstName, LastName, City FROM customers WHERE Country = 'Germany'")

4 Ergebnisse gefunden.
------------------------------------------------------------
('Leonie', 'Köhler', 'Stuttgart')
('Hannah', 'Schneider', 'Berlin')
('Fynn', 'Zimmermann', 'Frankfurt')
('Niklas', 'Schröder', 'Berlin')


[('Leonie', 'Köhler', 'Stuttgart'),
 ('Hannah', 'Schneider', 'Berlin'),
 ('Fynn', 'Zimmermann', 'Frankfurt'),
 ('Niklas', 'Schröder', 'Berlin')]

**Wichtig:** Bei Textvergleichen wird standardmäßig die Groß-/Kleinschreibung beachtet. `'Germany'` und `'germany'` sind unterschiedliche Werte.

### Übung 1.1: Einfache WHERE-Abfrage

**Aufgabe:** Schreiben Sie eine Abfrage, die alle Tracks mit einer Länge von mehr als 300.000 Millisekunden (5 Minuten) anzeigt. Geben Sie den Namen und die Länge in Millisekunden aus.

**Hinweise:**
- Die Spalte für die Länge heißt `Milliseconds`
- Verwenden Sie den Operator `>`
- Nutzen Sie die Funktion `query_db()`

In [4]:
# Ihr Code hier


**Musterlösung:**

<details>
<summary>Lösung anzeigen</summary>

```python
# Tracks länger als 5 Minuten (300.000 ms)
query_db("SELECT Name, Milliseconds FROM tracks WHERE Milliseconds > 300000")
```

**Erklärung:** Die WHERE-Klausel filtert alle Tracks, deren Milliseconds-Wert größer als 300.000 ist. Dies entspricht Tracks mit einer Länge von mehr als 5 Minuten.
</details>

---

### Komplexe Bedingungen mit AND, OR und NOT

Häufig müssen mehrere Bedingungen kombiniert werden. SQL bietet dafür die logischen Operatoren `AND`, `OR` und `NOT`, die Sie bereits aus Notebook 08 (Operatoren) kennen.

**Logische Operatoren:**

| Operator | Bedeutung | Ergebnis |
|----------|-----------|----------|
| `AND` | Beide Bedingungen müssen wahr sein | Nur wenn A UND B wahr |
| `OR` | Mindestens eine Bedingung muss wahr sein | Wenn A ODER B wahr |
| `NOT` | Kehrt die Bedingung um | Wenn A NICHT wahr |

### Beispiel 3: Bedingungen mit AND

Wir suchen günstige, kurze Tracks – Songs unter 1€ UND unter 3 Minuten:

In [ ]:
# Günstige UND kurze Tracks
query_db("""
    SELECT Name, UnitPrice, Milliseconds 
    FROM tracks 
    WHERE UnitPrice < 1 AND Milliseconds < 180000
""")

479 Ergebnisse gefunden.
------------------------------------------------------------
('Right Through You', 0.99, 176117)
('We Die Young', 0.99, 152084)
('Samba De Uma Nota Só (One Note Samba)', 0.99, 137273)
('Por Causa De Você', 0.99, 169900)
('Fotografia', 0.99, 129227)
('Se Todos Fossem Iguais A Você (Instrumental)', 0.99, 134948)
('Angela', 0.99, 169508)
('Outra Vez', 0.99, 126511)
('Money', 0.99, 147591)
('Long Tall Sally', 0.99, 106396)
... und 469 weitere Ergebnisse


[('Right Through You', 0.99, 176117),
 ('We Die Young', 0.99, 152084),
 ('Samba De Uma Nota Só (One Note Samba)', 0.99, 137273),
 ('Por Causa De Você', 0.99, 169900),
 ('Fotografia', 0.99, 129227),
 ('Se Todos Fossem Iguais A Você (Instrumental)', 0.99, 134948),
 ('Angela', 0.99, 169508),
 ('Outra Vez', 0.99, 126511),
 ('Money', 0.99, 147591),
 ('Long Tall Sally', 0.99, 106396),
 ('Bad Boy', 0.99, 116088),
 ('Twist And Shout', 0.99, 161123),
 ('Please Mr. Postman', 0.99, 137639),
 ("C'Mon Everybody", 0.99, 140199),
 ("Rock 'N' Roll Music", 0.99, 141923),
 ('Slow Down', 0.99, 163265),
 ('Roadrunner', 0.99, 143595),
 ('Carol', 0.99, 143830),
 ('Good Golly Miss Molly', 0.99, 106266),
 ('20 Flight Rock', 0.99, 107807),
 ('FX', 0.99, 103157),
 ('Laguna Sunrise', 0.99, 173087),
 ('St. Vitus Dance', 0.99, 149655),
 ('Smoked Pork', 0.99, 47333),
 ('Now Sports', 0.99, 4884),
 ('A Statistic', 0.99, 6373),
 ('The Real Problem', 0.99, 11650),
 ('KKK Bitch', 0.99, 173008),
 ('D Note', 0.99, 95738),

In [9]:
# Günstige UND kurze Tracks
query_db("""
    SELECT Name, UnitPrice 
    FROM tracks 
    WHERE UnitPrice < 1 AND Milliseconds < 180000
""")

479 Ergebnisse gefunden.
------------------------------------------------------------
('Right Through You', 0.99)
('We Die Young', 0.99)
('Samba De Uma Nota Só (One Note Samba)', 0.99)
('Por Causa De Você', 0.99)
('Fotografia', 0.99)
('Se Todos Fossem Iguais A Você (Instrumental)', 0.99)
('Angela', 0.99)
('Outra Vez', 0.99)
('Money', 0.99)
('Long Tall Sally', 0.99)
... und 469 weitere Ergebnisse


[('Right Through You', 0.99),
 ('We Die Young', 0.99),
 ('Samba De Uma Nota Só (One Note Samba)', 0.99),
 ('Por Causa De Você', 0.99),
 ('Fotografia', 0.99),
 ('Se Todos Fossem Iguais A Você (Instrumental)', 0.99),
 ('Angela', 0.99),
 ('Outra Vez', 0.99),
 ('Money', 0.99),
 ('Long Tall Sally', 0.99),
 ('Bad Boy', 0.99),
 ('Twist And Shout', 0.99),
 ('Please Mr. Postman', 0.99),
 ("C'Mon Everybody", 0.99),
 ("Rock 'N' Roll Music", 0.99),
 ('Slow Down', 0.99),
 ('Roadrunner', 0.99),
 ('Carol', 0.99),
 ('Good Golly Miss Molly', 0.99),
 ('20 Flight Rock', 0.99),
 ('FX', 0.99),
 ('Laguna Sunrise', 0.99),
 ('St. Vitus Dance', 0.99),
 ('Smoked Pork', 0.99),
 ('Now Sports', 0.99),
 ('A Statistic', 0.99),
 ('The Real Problem', 0.99),
 ('KKK Bitch', 0.99),
 ('D Note', 0.99),
 ('Oprah', 0.99),
 ('Body Count Anthem', 0.99),
 ('First Time I Met The Blues', 0.99),
 ('Let Me Love You Baby', 0.99),
 ('She Suits Me To A Tee', 0.99),
 ('Keep It To Myself (Aka Keep It To Yourself)', 0.99),
 ('Too Many Wa

Mit `AND` werden nur Datensätze zurückgegeben, die **beide** Bedingungen erfüllen.

### Beispiel 4: Bedingungen mit OR

Wir suchen Kunden aus Deutschland ODER Frankreich:

In [ ]:
# Kunden aus Deutschland ODER Frankreich
query_db("""
    SELECT FirstName, LastName, Country 
    FROM customers 
    WHERE Country = 'Germany' OR Country = 'France'
""")

9 Ergebnisse gefunden.
------------------------------------------------------------
('Leonie', 'Köhler', 'Germany')
('Hannah', 'Schneider', 'Germany')
('Fynn', 'Zimmermann', 'Germany')
('Niklas', 'Schröder', 'Germany')
('Camille', 'Bernard', 'France')
('Dominique', 'Lefebvre', 'France')
('Marc', 'Dubois', 'France')
('Wyatt', 'Girard', 'France')
('Isabelle', 'Mercier', 'France')


[('Leonie', 'Köhler', 'Germany'),
 ('Hannah', 'Schneider', 'Germany'),
 ('Fynn', 'Zimmermann', 'Germany'),
 ('Niklas', 'Schröder', 'Germany'),
 ('Camille', 'Bernard', 'France'),
 ('Dominique', 'Lefebvre', 'France'),
 ('Marc', 'Dubois', 'France'),
 ('Wyatt', 'Girard', 'France'),
 ('Isabelle', 'Mercier', 'France')]

Mit `OR` werden Datensätze zurückgegeben, die **mindestens eine** der Bedingungen erfüllen.

### Übung 1.2: Komplexe Bedingungen

**Aufgabe:** Finden Sie alle Tracks, die entweder zum Genre 1 (Rock) ODER zum Genre 3 (Metal) gehören UND einen Preis von 0.99€ haben.

**Hinweise:**
- Verwenden Sie Klammern zur Gruppierung: `(bedingung1 OR bedingung2) AND bedingung3`
- Die Genre-Spalte heißt `GenreId`

In [7]:
# Ihr Code hier


**Musterlösung:**

<details>
<summary>Lösung anzeigen</summary>

```python
# Rock oder Metal Tracks für 0.99€
query_db("""
    SELECT Name, GenreId, UnitPrice 
    FROM tracks 
    WHERE (GenreId = 1 OR GenreId = 3) AND UnitPrice = 0.99
""")
```

**Erklärung:** Die Klammern stellen sicher, dass zuerst die OR-Bedingung ausgewertet wird. Ohne Klammern würde `GenreId = 1 OR (GenreId = 3 AND UnitPrice = 0.99)` ausgewertet – ein völlig anderes Ergebnis!
</details>

---

### Spezielle Operatoren: BETWEEN, IN und LIKE

SQL bietet spezielle Operatoren für häufige Filterszenarien:

| Operator | Verwendung | Beispiel |
|----------|------------|----------|
| `BETWEEN` | Wertebereich (inklusiv) | `WHERE preis BETWEEN 1 AND 5` |
| `IN` | Liste von Werten | `WHERE land IN ('DE', 'AT', 'CH')` |
| `LIKE` | Musterabgleich | `WHERE name LIKE 'A%'` |

### Beispiel 5: BETWEEN für Wertebereiche

Tracks mit einer Länge zwischen 3 und 4 Minuten:

In [10]:
# Tracks zwischen 3 und 4 Minuten
query_db("""
    SELECT Name, Milliseconds 
    FROM tracks 
    WHERE Milliseconds BETWEEN 180000 AND 240000
""")

982 Ergebnisse gefunden.
------------------------------------------------------------
('Fast As a Shark', 230619)
('Put The Finger On You', 205662)
("Let's Get It Up", 233926)
('Inject The Venom', 210834)
('Snowballed', 203102)
('C.O.D.', 199836)
('Night Of The Long Knives', 205688)
('Dog Eat Dog', 215196)
('Deuces Are Wild', 215875)
('Perfect', 188133)
... und 972 weitere Ergebnisse


[('Fast As a Shark', 230619),
 ('Put The Finger On You', 205662),
 ("Let's Get It Up", 233926),
 ('Inject The Venom', 210834),
 ('Snowballed', 203102),
 ('C.O.D.', 199836),
 ('Night Of The Long Knives', 205688),
 ('Dog Eat Dog', 215196),
 ('Deuces Are Wild', 215875),
 ('Perfect', 188133),
 ('Hand In My Pocket', 221570),
 ('You Learn', 239699),
 ('Ironic', 229825),
 ('Not The Doctor', 227631),
 ("I Can't Remember", 222955),
 ('Put You Down', 196231),
 ('Desafinado', 185338),
 ('Falando De Amor', 219663),
 ('Corcovado (Quiet Nights Of Quiet Stars)', 205662),
 ('Enter Sandman', 221701),
 ('Cochise', 222380),
 ('Exploder', 206053),
 ('Hypnotize', 206628),
 ('Drown Me Slowly', 233691),
 ('The Worm', 237714),
 ('Man Or Animal', 233195),
 ('All For You', 235833),
 ('Heart Of Gold', 194873),
 ('Behind The Wall Of Sleep', 217573),
 ('Evil Woman', 204930),
 ('Warning', 212062),
 ("Tomorrow's Dream", 192496),
 ('Cornucopia', 234814),
 ("Body Count's In The House", 204251),
 ('Bowels Of The Devil'

`BETWEEN` ist inklusiv – die Grenzen sind im Ergebnis enthalten. Die Abfrage entspricht: `WHERE Milliseconds >= 180000 AND Milliseconds <= 240000`

### Beispiel 6: IN für Wertelisten

Kunden aus deutschsprachigen Ländern:

In [12]:
# Kunden aus Deutschland, Österreich oder Schweiz
query_db("""
    SELECT FirstName, LastName, Country 
    FROM customers 
    WHERE Country IN ('Germany', 'France')
""")

9 Ergebnisse gefunden.
------------------------------------------------------------
('Leonie', 'Köhler', 'Germany')
('Hannah', 'Schneider', 'Germany')
('Fynn', 'Zimmermann', 'Germany')
('Niklas', 'Schröder', 'Germany')
('Camille', 'Bernard', 'France')
('Dominique', 'Lefebvre', 'France')
('Marc', 'Dubois', 'France')
('Wyatt', 'Girard', 'France')
('Isabelle', 'Mercier', 'France')


[('Leonie', 'Köhler', 'Germany'),
 ('Hannah', 'Schneider', 'Germany'),
 ('Fynn', 'Zimmermann', 'Germany'),
 ('Niklas', 'Schröder', 'Germany'),
 ('Camille', 'Bernard', 'France'),
 ('Dominique', 'Lefebvre', 'France'),
 ('Marc', 'Dubois', 'France'),
 ('Wyatt', 'Girard', 'France'),
 ('Isabelle', 'Mercier', 'France')]

`IN` ist eine elegante Alternative zu mehreren OR-Bedingungen und macht die Abfrage lesbarer.

### Beispiel 7: LIKE für Musterabgleiche

Der LIKE-Operator ermöglicht die Suche nach Textmustern:
- `%` steht für beliebig viele Zeichen (0 oder mehr)
- `_` steht für genau ein Zeichen

In [14]:
# Tracks, deren Name mit "Love" beginnt
query_db("SELECT Name FROM tracks WHERE Name LIKE '%Love%'")

114 Ergebnisse gefunden.
------------------------------------------------------------
('Love In An Elevator',)
('Love, Hate, Love',)
('Let Me Love You Baby',)
('My Love',)
('The Girl I Love She Got Long Black Wavy Hair',)
('Whole Lotta Love',)
('Loverman',)
('Love Gun',)
('Do You Love Me',)
('Calling Dr. Love',)
... und 104 weitere Ergebnisse


[('Love In An Elevator',),
 ('Love, Hate, Love',),
 ('Let Me Love You Baby',),
 ('My Love',),
 ('The Girl I Love She Got Long Black Wavy Hair',),
 ('Whole Lotta Love',),
 ('Loverman',),
 ('Love Gun',),
 ('Do You Love Me',),
 ('Calling Dr. Love',),
 ('Love Is Blind',),
 ('Cry For Love',),
 ('Living On Love',),
 ('Love Of My Life',),
 ('Um Love',),
 ('Do You Have Other Loves?',),
 ("Don't Take Your Love From Me",),
 ('I Need Love',),
 ('Love Child',),
 ("Cascades : I'm Not Your Lover",),
 ('Love Conquers All',),
 ("Love Don't Mean a Thing",),
 ("You Can't Do it Right (With the One You Love)",),
 ('Talk About Love',),
 ('Love Bites',),
 ('When Love & Hate Collide',),
 ('Make Love Like A Man',),
 ('Sunshine Of Your Love',),
 ('Old Love',),
 ('She Loves Me Not',),
 ('Underwater Love',),
 ('What Now My Love',),
 ('Summer Love',),
 ('Love And Marriage',),
 ('Loves Been Good To Me',),
 ('Is This Love (Live)',),
 ("Jesus Of Suburbia / City Of The Damned / I Don't Care / Dearly Beloved / Tales O

In [11]:
# Tracks, die "Love" irgendwo im Namen enthalten
query_db("SELECT Name FROM tracks WHERE Name LIKE '%Love%'")

114 Ergebnisse gefunden.
------------------------------------------------------------
('Love In An Elevator',)
('Love, Hate, Love',)
('Let Me Love You Baby',)
('My Love',)
('The Girl I Love She Got Long Black Wavy Hair',)
('Whole Lotta Love',)
('Loverman',)
('Love Gun',)
('Do You Love Me',)
('Calling Dr. Love',)
... und 104 weitere Ergebnisse


[('Love In An Elevator',),
 ('Love, Hate, Love',),
 ('Let Me Love You Baby',),
 ('My Love',),
 ('The Girl I Love She Got Long Black Wavy Hair',),
 ('Whole Lotta Love',),
 ('Loverman',),
 ('Love Gun',),
 ('Do You Love Me',),
 ('Calling Dr. Love',),
 ('Love Is Blind',),
 ('Cry For Love',),
 ('Living On Love',),
 ('Love Of My Life',),
 ('Um Love',),
 ('Do You Have Other Loves?',),
 ("Don't Take Your Love From Me",),
 ('I Need Love',),
 ('Love Child',),
 ("Cascades : I'm Not Your Lover",),
 ('Love Conquers All',),
 ("Love Don't Mean a Thing",),
 ("You Can't Do it Right (With the One You Love)",),
 ('Talk About Love',),
 ('Love Bites',),
 ('When Love & Hate Collide',),
 ('Make Love Like A Man',),
 ('Sunshine Of Your Love',),
 ('Old Love',),
 ('She Loves Me Not',),
 ('Underwater Love',),
 ('What Now My Love',),
 ('Summer Love',),
 ('Love And Marriage',),
 ('Loves Been Good To Me',),
 ('Is This Love (Live)',),
 ("Jesus Of Suburbia / City Of The Damned / I Don't Care / Dearly Beloved / Tales O

---

### NULL-Werte prüfen

In Notebook 21 haben Sie gelernt, dass `NULL` in Datenbanken "kein Wert" bedeutet. NULL-Werte können **nicht** mit `=` verglichen werden – Sie müssen `IS NULL` oder `IS NOT NULL` verwenden.

**Wichtig:** `WHERE spalte = NULL` funktioniert NICHT! Verwenden Sie stattdessen `WHERE spalte IS NULL`.

In [12]:
# Tracks ohne Komponisten-Angabe
query_db("SELECT Name, Composer FROM tracks WHERE Composer IS NULL")

978 Ergebnisse gefunden.
------------------------------------------------------------
('Balls to the Wall', None)
('Desafinado', None)
('Garota De Ipanema', None)
('Samba De Uma Nota Só (One Note Samba)', None)
('Por Causa De Você', None)
('Ligia', None)
('Fotografia', None)
('Dindi (Dindi)', None)
('Se Todos Fossem Iguais A Você (Instrumental)', None)
('Falando De Amor', None)
... und 968 weitere Ergebnisse


[('Balls to the Wall', None),
 ('Desafinado', None),
 ('Garota De Ipanema', None),
 ('Samba De Uma Nota Só (One Note Samba)', None),
 ('Por Causa De Você', None),
 ('Ligia', None),
 ('Fotografia', None),
 ('Dindi (Dindi)', None),
 ('Se Todos Fossem Iguais A Você (Instrumental)', None),
 ('Falando De Amor', None),
 ('Angela', None),
 ('Corcovado (Quiet Nights Of Quiet Stars)', None),
 ('Outra Vez', None),
 ('O Boto (Bôto)', None),
 ('Canta, Canta Mais', None),
 ('Intro/ Low Down', None),
 ('13 Years Of Grief', None),
 ('Stronger Than Death', None),
 ('All For You', None),
 ('Super Terrorizer', None),
 ('Phoney Smile Fake Hellos', None),
 ('Lost My Better Half', None),
 ('Bored To Tears', None),
 ('A.N.D.R.O.T.A.Z.', None),
 ('Born To Booze', None),
 ('World Of Trouble', None),
 ('No More Tears', None),
 ('The Begining... At Last', None),
 ('Heart Of Gold', None),
 ('Snowblind', None),
 ('Like A Bird', None),
 ('Blood In The Wall', None),
 ('The Beginning...At Last', None),
 ('Black Sabb

In [13]:
# Tracks MIT Komponisten-Angabe
query_db("SELECT Name, Composer FROM tracks WHERE Composer IS NOT NULL")

2525 Ergebnisse gefunden.
------------------------------------------------------------
('For Those About To Rock (We Salute You)', 'Angus Young, Malcolm Young, Brian Johnson')
('Fast As a Shark', 'F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman')
('Restless and Wild', 'F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. Dirkscneider & W. Hoffman')
('Princess of the Dawn', 'Deaffy & R.A. Smith-Diesel')
('Put The Finger On You', 'Angus Young, Malcolm Young, Brian Johnson')
("Let's Get It Up", 'Angus Young, Malcolm Young, Brian Johnson')
('Inject The Venom', 'Angus Young, Malcolm Young, Brian Johnson')
('Snowballed', 'Angus Young, Malcolm Young, Brian Johnson')
('Evil Walks', 'Angus Young, Malcolm Young, Brian Johnson')
('C.O.D.', 'Angus Young, Malcolm Young, Brian Johnson')
... und 2515 weitere Ergebnisse


[('For Those About To Rock (We Salute You)',
  'Angus Young, Malcolm Young, Brian Johnson'),
 ('Fast As a Shark', 'F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman'),
 ('Restless and Wild',
  'F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. Dirkscneider & W. Hoffman'),
 ('Princess of the Dawn', 'Deaffy & R.A. Smith-Diesel'),
 ('Put The Finger On You', 'Angus Young, Malcolm Young, Brian Johnson'),
 ("Let's Get It Up", 'Angus Young, Malcolm Young, Brian Johnson'),
 ('Inject The Venom', 'Angus Young, Malcolm Young, Brian Johnson'),
 ('Snowballed', 'Angus Young, Malcolm Young, Brian Johnson'),
 ('Evil Walks', 'Angus Young, Malcolm Young, Brian Johnson'),
 ('C.O.D.', 'Angus Young, Malcolm Young, Brian Johnson'),
 ('Breaking The Rules', 'Angus Young, Malcolm Young, Brian Johnson'),
 ('Night Of The Long Knives', 'Angus Young, Malcolm Young, Brian Johnson'),
 ('Spellbound', 'Angus Young, Malcolm Young, Brian Johnson'),
 ('Go Down', 'AC/DC'),
 ('Dog Eat Dog', 'AC/DC'),
 ('Let There Be Rock', 'A

---

## Kapitel 2: Ergebnisse sortieren mit ORDER BY

### Einführung und Motivation

Die Reihenfolge der Ergebnisse einer SQL-Abfrage ist standardmäßig nicht definiert – die Datenbank gibt die Daten in der internen Speicherreihenfolge zurück. Für aussagekräftige Analysen und Berichte benötigen Sie jedoch oft sortierte Daten: alphabetisch nach Namen, chronologisch nach Datum oder nach Preis geordnet.

Die `ORDER BY`-Klausel ermöglicht die Sortierung nach einer oder mehreren Spalten in aufsteigender (`ASC`) oder absteigender (`DESC`) Reihenfolge.

### Syntax von ORDER BY

```sql
SELECT spalten
FROM tabelle
WHERE bedingung
ORDER BY spalte [ASC|DESC];
```

- `ASC` (ascending): Aufsteigende Sortierung (Standard, kann weggelassen werden)
- `DESC` (descending): Absteigende Sortierung

### Beispiel 8: Aufsteigende Sortierung

In [16]:
# Kunden alphabetisch nach Nachname sortiert
query_db("""
    SELECT FirstName, LastName, Country 
    FROM customers 
    ORDER BY LastName ASC
""")

59 Ergebnisse gefunden.
------------------------------------------------------------
('Roberto', 'Almeida', 'Brazil')
('Julia', 'Barnett', 'USA')
('Camille', 'Bernard', 'France')
('Michelle', 'Brooks', 'USA')
('Robert', 'Brown', 'Canada')
('Kathy', 'Chase', 'USA')
('Richard', 'Cunningham', 'USA')
('Marc', 'Dubois', 'France')
('João', 'Fernandes', 'Portugal')
('Edward', 'Francis', 'Canada')
... und 49 weitere Ergebnisse


[('Roberto', 'Almeida', 'Brazil'),
 ('Julia', 'Barnett', 'USA'),
 ('Camille', 'Bernard', 'France'),
 ('Michelle', 'Brooks', 'USA'),
 ('Robert', 'Brown', 'Canada'),
 ('Kathy', 'Chase', 'USA'),
 ('Richard', 'Cunningham', 'USA'),
 ('Marc', 'Dubois', 'France'),
 ('João', 'Fernandes', 'Portugal'),
 ('Edward', 'Francis', 'Canada'),
 ('Wyatt', 'Girard', 'France'),
 ('Luís', 'Gonçalves', 'Brazil'),
 ('John', 'Gordon', 'USA'),
 ('Tim', 'Goyer', 'USA'),
 ('Patrick', 'Gray', 'USA'),
 ('Astrid', 'Gruber', 'Austria'),
 ('Diego', 'Gutiérrez', 'Argentina'),
 ('Bjørn', 'Hansen', 'Norway'),
 ('Frank', 'Harris', 'USA'),
 ('Helena', 'Holý', 'Czech Republic'),
 ('Phil', 'Hughes', 'United Kingdom'),
 ('Terhi', 'Hämäläinen', 'Finland'),
 ('Joakim', 'Johansson', 'Sweden'),
 ('Emma', 'Jones', 'United Kingdom'),
 ('Ladislav', 'Kovács', 'Hungary'),
 ('Leonie', 'Köhler', 'Germany'),
 ('Heather', 'Leacock', 'USA'),
 ('Dominique', 'Lefebvre', 'France'),
 ('Lucas', 'Mancini', 'Italy'),
 ('Eduardo', 'Martins', 'Braz

### Beispiel 9: Absteigende Sortierung

In [19]:
# Längste Tracks zuerst
query_db("""
    SELECT Name, Milliseconds 
    FROM tracks 
    ORDER BY Milliseconds ASC
""")

3503 Ergebnisse gefunden.
------------------------------------------------------------
('É Uma Partida De Futebol', 1071)
('Now Sports', 4884)
('A Statistic', 6373)
('Oprah', 6635)
('Commercial 1', 7941)
('The Real Problem', 11650)
('Commercial 2', 21211)
('Bossa', 29048)
('Casinha Feliz', 32287)
('Mateus Enter', 33149)
... und 3493 weitere Ergebnisse


[('É Uma Partida De Futebol', 1071),
 ('Now Sports', 4884),
 ('A Statistic', 6373),
 ('Oprah', 6635),
 ('Commercial 1', 7941),
 ('The Real Problem', 11650),
 ('Commercial 2', 21211),
 ('Bossa', 29048),
 ('Casinha Feliz', 32287),
 ('Mateus Enter', 33149),
 ('Deixa Entrar', 33619),
 ('Homem Primata (Vinheta)', 34168),
 ('Cabeça Dinossauro', 37120),
 ('Freedom For My People', 38164),
 ('Demorou!', 39131),
 ('The Hellion', 41900),
 ('Little Guitars (Intro)', 42240),
 ('The Star Spangled Banner', 43232),
 ('Blanco', 45191),
 ('Smoked Pork', 47333),
 ('Intro- Churchill S Speech', 48013),
 ('Intro', 49737),
 ('Étude 1, In C Major - Preludio (Presto) - Liszt', 51780),
 ('Intro', 52218),
 ('Wasted Reprise', 53733),
 ('Cotidiano N 2', 55902),
 ('Polícia (Vinheta)', 56111),
 ('Soldier Side - Intro', 63764),
 ('Happy Trails', 65488),
 ('Arc', 65593),
 ("L'orfeo, Act 3, Sinfonia (Orchestra)", 66639),
 ('Salutations', 69120),
 ('Lamentations of Jeremiah, First Set \\ Incipit Lamentatio', 69194),
 ('

### Beispiel 10: Sortierung nach mehreren Spalten

Sie können nach mehreren Spalten sortieren – die erste Spalte hat Priorität:

In [16]:
# Kunden nach Land, dann nach Stadt sortiert
query_db("""
    SELECT Country, City, FirstName, LastName 
    FROM customers 
    ORDER BY Country ASC, City ASC
""")

59 Ergebnisse gefunden.
------------------------------------------------------------
('Argentina', 'Buenos Aires', 'Diego', 'Gutiérrez')
('Australia', 'Sidney', 'Mark', 'Taylor')
('Austria', 'Vienne', 'Astrid', 'Gruber')
('Belgium', 'Brussels', 'Daan', 'Peeters')
('Brazil', 'Brasília', 'Fernanda', 'Ramos')
('Brazil', 'Rio de Janeiro', 'Roberto', 'Almeida')
('Brazil', 'São José dos Campos', 'Luís', 'Gonçalves')
('Brazil', 'São Paulo', 'Eduardo', 'Martins')
('Brazil', 'São Paulo', 'Alexandre', 'Rocha')
('Canada', 'Edmonton', 'Mark', 'Philips')
... und 49 weitere Ergebnisse


[('Argentina', 'Buenos Aires', 'Diego', 'Gutiérrez'),
 ('Australia', 'Sidney', 'Mark', 'Taylor'),
 ('Austria', 'Vienne', 'Astrid', 'Gruber'),
 ('Belgium', 'Brussels', 'Daan', 'Peeters'),
 ('Brazil', 'Brasília', 'Fernanda', 'Ramos'),
 ('Brazil', 'Rio de Janeiro', 'Roberto', 'Almeida'),
 ('Brazil', 'São José dos Campos', 'Luís', 'Gonçalves'),
 ('Brazil', 'São Paulo', 'Eduardo', 'Martins'),
 ('Brazil', 'São Paulo', 'Alexandre', 'Rocha'),
 ('Canada', 'Edmonton', 'Mark', 'Philips'),
 ('Canada', 'Halifax', 'Martha', 'Silk'),
 ('Canada', 'Montréal', 'François', 'Tremblay'),
 ('Canada', 'Ottawa', 'Edward', 'Francis'),
 ('Canada', 'Toronto', 'Robert', 'Brown'),
 ('Canada', 'Vancouver', 'Jennifer', 'Peterson'),
 ('Canada', 'Winnipeg', 'Aaron', 'Mitchell'),
 ('Canada', 'Yellowknife', 'Ellie', 'Sullivan'),
 ('Chile', 'Santiago', 'Luis', 'Rojas'),
 ('Czech Republic', 'Prague', 'František', 'Wichterlová'),
 ('Czech Republic', 'Prague', 'Helena', 'Holý'),
 ('Denmark', 'Copenhagen', 'Kara', 'Nielsen')

### Übung 2.1: Sortierung anwenden

**Aufgabe:** Zeigen Sie die 10 günstigsten Tracks an. Geben Sie Name und Preis aus, sortiert nach Preis aufsteigend.

**Hinweise:**
- Nutzen Sie ORDER BY für die Sortierung
- Die Funktion query_db zeigt standardmäßig nur 10 Ergebnisse

In [17]:
# Ihr Code hier


**Musterlösung:**

<details>
<summary>Lösung anzeigen</summary>

```python
# Die 10 günstigsten Tracks
query_db("""
    SELECT Name, UnitPrice 
    FROM tracks 
    ORDER BY UnitPrice ASC
""")
```

**Erklärung:** ORDER BY UnitPrice ASC sortiert die Tracks nach Preis, beginnend mit dem günstigsten. Die query_db-Funktion zeigt automatisch nur die ersten 10 Ergebnisse.
</details>

---

## Kapitel 3: Tabellen verbinden mit JOIN

### Einführung und Motivation

In Notebook 21 haben Sie gelernt, dass relationale Datenbanken Daten in mehreren verknüpften Tabellen speichern, um Redundanzen zu vermeiden. Die Künstler-Informationen stehen in der `artists`-Tabelle, die Alben in der `albums`-Tabelle. Um beide Informationen gemeinsam abzufragen – etwa "Welche Alben hat welcher Künstler?" – müssen die Tabellen verknüpft werden.

Der `JOIN`-Befehl ermöglicht genau das: Er verbindet Zeilen aus zwei oder mehr Tabellen basierend auf einer gemeinsamen Spalte (typischerweise einem Fremdschlüssel). Dies ist eines der mächtigsten Konzepte in SQL und essentiell für die Arbeit mit relationalen Datenbanken.

SQLite unterstützt verschiedene JOIN-Typen. Die wichtigsten sind:
- **INNER JOIN**: Gibt nur Zeilen zurück, die in beiden Tabellen eine Übereinstimmung haben
- **LEFT JOIN**: Gibt alle Zeilen der linken Tabelle zurück, auch ohne Übereinstimmung

### INNER JOIN: Nur übereinstimmende Zeilen

**Syntax:**
```sql
SELECT spalten
FROM tabelle1
INNER JOIN tabelle2 ON tabelle1.spalte = tabelle2.spalte;
```

Der INNER JOIN gibt nur Zeilen zurück, bei denen die Verknüpfungsbedingung (nach `ON`) erfüllt ist.

### Beispiel 11: INNER JOIN zwischen albums und artists

Wir möchten wissen, welche Alben zu welchen Künstlern gehören:

In [18]:
# Alben mit Künstlernamen
query_db("""
    SELECT albums.Title, artists.Name
    FROM albums
    INNER JOIN artists ON albums.ArtistId = artists.ArtistId
""")

347 Ergebnisse gefunden.
------------------------------------------------------------
('For Those About To Rock We Salute You', 'AC/DC')
('Balls to the Wall', 'Accept')
('Restless and Wild', 'Accept')
('Let There Be Rock', 'AC/DC')
('Big Ones', 'Aerosmith')
('Jagged Little Pill', 'Alanis Morissette')
('Facelift', 'Alice In Chains')
('Warner 25 Anos', 'Antônio Carlos Jobim')
('Plays Metallica By Four Cellos', 'Apocalyptica')
('Audioslave', 'Audioslave')
... und 337 weitere Ergebnisse


[('For Those About To Rock We Salute You', 'AC/DC'),
 ('Balls to the Wall', 'Accept'),
 ('Restless and Wild', 'Accept'),
 ('Let There Be Rock', 'AC/DC'),
 ('Big Ones', 'Aerosmith'),
 ('Jagged Little Pill', 'Alanis Morissette'),
 ('Facelift', 'Alice In Chains'),
 ('Warner 25 Anos', 'Antônio Carlos Jobim'),
 ('Plays Metallica By Four Cellos', 'Apocalyptica'),
 ('Audioslave', 'Audioslave'),
 ('Out Of Exile', 'Audioslave'),
 ('BackBeat Soundtrack', 'BackBeat'),
 ('The Best Of Billy Cobham', 'Billy Cobham'),
 ('Alcohol Fueled Brewtality Live! [Disc 1]', 'Black Label Society'),
 ('Alcohol Fueled Brewtality Live! [Disc 2]', 'Black Label Society'),
 ('Black Sabbath', 'Black Sabbath'),
 ('Black Sabbath Vol. 4 (Remaster)', 'Black Sabbath'),
 ('Body Count', 'Body Count'),
 ('Chemical Wedding', 'Bruce Dickinson'),
 ('The Best Of Buddy Guy - The Millenium Collection', 'Buddy Guy'),
 ('Prenda Minha', 'Caetano Veloso'),
 ('Sozinho Remix Ao Vivo', 'Caetano Veloso'),
 ('Minha Historia', 'Chico Buarque'

**Erklärung des Beispiels:**

1. `FROM albums`: Wir starten mit der albums-Tabelle
2. `INNER JOIN artists`: Wir verbinden mit der artists-Tabelle
3. `ON albums.ArtistId = artists.ArtistId`: Die Verknüpfung erfolgt über die ArtistId
4. `SELECT albums.Title, artists.Name`: Wir wählen die gewünschten Spalten aus beiden Tabellen

### Beispiel 12: INNER JOIN mit WHERE-Filter

JOINs können mit WHERE kombiniert werden:

In [19]:
# Alben des Künstlers "AC/DC"
query_db("""
    SELECT albums.Title, artists.Name
    FROM albums
    INNER JOIN artists ON albums.ArtistId = artists.ArtistId
    WHERE artists.Name = 'AC/DC'
""")

2 Ergebnisse gefunden.
------------------------------------------------------------
('For Those About To Rock We Salute You', 'AC/DC')
('Let There Be Rock', 'AC/DC')


[('For Those About To Rock We Salute You', 'AC/DC'),
 ('Let There Be Rock', 'AC/DC')]

### Übung 3.1: INNER JOIN anwenden

**Aufgabe:** Schreiben Sie eine Abfrage, die alle Tracks mit ihrem Genrenamen anzeigt. Die Tabelle `genres` enthält die Spalten GenreId und Name.

**Hinweise:**
- Verbinden Sie `tracks` und `genres` über `GenreId`
- Geben Sie den Track-Namen und den Genre-Namen aus
- Achten Sie auf eindeutige Spaltennamen: `tracks.Name` und `genres.Name`

In [20]:
# Ihr Code hier


**Musterlösung:**

<details>
<summary>Lösung anzeigen</summary>

```python
# Tracks mit Genre-Namen
query_db("""
    SELECT tracks.Name, genres.Name
    FROM tracks
    INNER JOIN genres ON tracks.GenreId = genres.GenreId
""")
```

**Erklärung:** Der JOIN verbindet jeden Track mit seinem Genre über die gemeinsame GenreId. Da beide Tabellen eine Spalte "Name" haben, müssen wir die Tabellennamen als Präfix verwenden.
</details>

---

### LEFT JOIN: Alle Zeilen der linken Tabelle

Der **LEFT JOIN** gibt alle Zeilen der linken Tabelle zurück, auch wenn es keine Übereinstimmung in der rechten Tabelle gibt. Für nicht übereinstimmende Zeilen werden NULL-Werte eingesetzt.

**Syntax:**
```sql
SELECT spalten
FROM tabelle1
LEFT JOIN tabelle2 ON tabelle1.spalte = tabelle2.spalte;
```

### Beispiel 13: LEFT JOIN – Künstler mit und ohne Alben

In [21]:
# Alle Künstler, auch ohne Alben
query_db("""
    SELECT artists.Name, albums.Title
    FROM artists
    LEFT JOIN albums ON artists.ArtistId = albums.ArtistId
    ORDER BY artists.Name
""")

418 Ergebnisse gefunden.
------------------------------------------------------------
('A Cor Do Som', None)
('AC/DC', 'For Those About To Rock We Salute You')
('AC/DC', 'Let There Be Rock')
('Aaron Copland & London Symphony Orchestra', 'A Copland Celebration, Vol. I')
('Aaron Goldberg', 'Worlds')
('Academy of St. Martin in the Fields & Sir Neville Marriner', 'The World of Classical Favourites')
('Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner', 'Sir Neville Marriner: A Celebration')
('Academy of St. Martin in the Fields, John Birch, Sir Neville Marriner & Sylvia McNair', 'Fauré: Requiem, Ravel: Pavane & Others')
('Academy of St. Martin in the Fields, Sir Neville Marriner & Thurston Dart', 'Bach: Orchestral Suites Nos. 1 - 4')
('Academy of St. Martin in the Fields, Sir Neville Marriner & William Bennett', None)
... und 408 weitere Ergebnisse


[('A Cor Do Som', None),
 ('AC/DC', 'For Those About To Rock We Salute You'),
 ('AC/DC', 'Let There Be Rock'),
 ('Aaron Copland & London Symphony Orchestra',
  'A Copland Celebration, Vol. I'),
 ('Aaron Goldberg', 'Worlds'),
 ('Academy of St. Martin in the Fields & Sir Neville Marriner',
  'The World of Classical Favourites'),
 ('Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner',
  'Sir Neville Marriner: A Celebration'),
 ('Academy of St. Martin in the Fields, John Birch, Sir Neville Marriner & Sylvia McNair',
  'Fauré: Requiem, Ravel: Pavane & Others'),
 ('Academy of St. Martin in the Fields, Sir Neville Marriner & Thurston Dart',
  'Bach: Orchestral Suites Nos. 1 - 4'),
 ('Academy of St. Martin in the Fields, Sir Neville Marriner & William Bennett',
  None),
 ('Accept', 'Balls to the Wall'),
 ('Accept', 'Restless and Wild'),
 ('Adrian Leaper & Doreen de Feis', 'Górecki: Symphony No. 3'),
 ('Aerosmith', 'Big Ones'),
 ("Aerosmith & Sierra Leone's Refugee Alls

Beachten Sie: Der erste Künstler "A Cor Do Som" hat `None` (NULL) als Album-Titel – er hat keine Alben in der Datenbank.

### Beispiel 14: LEFT JOIN – Nur Datensätze ohne Übereinstimmung finden

Mit LEFT JOIN und einer WHERE-Bedingung auf NULL können Sie Datensätze finden, die keine Verknüpfung haben:

In [22]:
# Künstler OHNE Alben
query_db("""
    SELECT artists.Name
    FROM artists
    LEFT JOIN albums ON artists.ArtistId = albums.ArtistId
    WHERE albums.AlbumId IS NULL
""")

71 Ergebnisse gefunden.
------------------------------------------------------------
('Milton Nascimento & Bebeto',)
('Azymuth',)
('João Gilberto',)
('Bebel Gilberto',)
('Jorge Vercilo',)
('Baby Consuelo',)
('Ney Matogrosso',)
('Luiz Melodia',)
('Nando Reis',)
('Pedro Luís & A Parede',)
... und 61 weitere Ergebnisse


[('Milton Nascimento & Bebeto',),
 ('Azymuth',),
 ('João Gilberto',),
 ('Bebel Gilberto',),
 ('Jorge Vercilo',),
 ('Baby Consuelo',),
 ('Ney Matogrosso',),
 ('Luiz Melodia',),
 ('Nando Reis',),
 ('Pedro Luís & A Parede',),
 ('Banda Black Rio',),
 ('Fernanda Porto',),
 ('Os Cariocas',),
 ('A Cor Do Som',),
 ('Kid Abelha',),
 ('Sandra De Sá',),
 ('Hermeto Pascoal',),
 ('Barão Vermelho',),
 ('Edson, DJ Marky & DJ Patife Featuring Fernanda Porto',),
 ('Santana Feat. Dave Matthews',),
 ('Santana Feat. Everlast',),
 ('Santana Feat. Rob Thomas',),
 ('Santana Feat. Lauryn Hill & Cee-Lo',),
 ('Santana Feat. The Project G&B',),
 ('Santana Feat. Maná',),
 ('Santana Feat. Eagle-Eye Cherry',),
 ('Santana Feat. Eric Clapton',),
 ('Vinícius De Moraes & Baden Powell',),
 ('Vinícius E Qurteto Em Cy',),
 ('Vinícius E Odette Lara',),
 ('Vinicius, Toquinho & Quarteto Em Cy',),
 ('Motörhead & Girlschool',),
 ('Peter Tosh',),
 ('R.E.M. Feat. KRS-One',),
 ('Simply Red',),
 ('Whitesnake',),
 ('Christina Aguil

### Übung 3.2: LEFT JOIN anwenden

**Aufgabe:** Finden Sie heraus, wie viele Kunden noch nie eine Rechnung erhalten haben. Nutzen Sie dafür einen LEFT JOIN zwischen `customers` und `invoices` und filtern Sie nach NULL-Werten.

**Hinweise:**
- Die Tabellen werden über `CustomerId` verknüpft
- Geben Sie Vorname, Nachname und InvoiceId aus
- Filtern Sie nach `invoices.InvoiceId IS NULL`

In [23]:
# Ihr Code hier


**Musterlösung:**

<details>
<summary>Lösung anzeigen</summary>

```python
# Kunden ohne Rechnungen
query_db("""
    SELECT customers.FirstName, customers.LastName, invoices.InvoiceId
    FROM customers
    LEFT JOIN invoices ON customers.CustomerId = invoices.CustomerId
    WHERE invoices.InvoiceId IS NULL
""")
```

**Erklärung:** Der LEFT JOIN zeigt alle Kunden, auch solche ohne Rechnungen. Die WHERE-Bedingung filtert dann nur die Kunden heraus, bei denen keine Rechnung existiert (InvoiceId ist NULL). In der Chinook-Datenbank haben alle Kunden Rechnungen, daher ist das Ergebnis leer.
</details>

---

## Kapitel 4: Daten ändern und löschen

### Einführung und Motivation

In Notebook 22 haben Sie gelernt, Daten mit `INSERT` einzufügen. Doch Daten müssen auch aktualisiert und gelöscht werden können. Ein Kunde zieht um und seine Adresse ändert sich – wir brauchen `UPDATE`. Ein Produkt wird aus dem Sortiment genommen – wir brauchen `DELETE`.

Diese Operationen sind **destruktiv** und können nicht rückgängig gemacht werden (ohne Backup). Daher ist besondere Vorsicht geboten, insbesondere bei der Verwendung der `WHERE`-Klausel.

### Hilfsfunktion für Schreiboperationen

Wir erstellen zunächst eine Hilfsfunktion für UPDATE- und DELETE-Operationen:

In [21]:
def execute_db(query, params=()):
    """Führt eine INSERT, UPDATE oder DELETE-Operation aus."""
    try:
        with sqlite3.connect('chinook.db') as conn:
            cursor = conn.cursor()
            cursor.execute(query, params)
            conn.commit()
            print(f"Erfolgreich: {cursor.rowcount} Zeile(n) betroffen.")
            return cursor.rowcount
    except sqlite3.Error as e:
        print(f"Fehler: {e}")
        return 0

---

### UPDATE: Daten aktualisieren

Der `UPDATE`-Befehl ändert bestehende Datensätze.

**Syntax:**
```sql
UPDATE tabelle
SET spalte1 = wert1, spalte2 = wert2, ...
WHERE bedingung;
```

**⚠️ WICHTIG:** Ohne `WHERE`-Klausel werden ALLE Zeilen der Tabelle aktualisiert!

### Beispiel 15: Einzelnen Datensatz aktualisieren

Wir fügen zunächst einen Testdatensatz ein und aktualisieren ihn dann:

In [22]:
# Neuen Künstler einfügen
execute_db("INSERT INTO artists (Name) VALUES (?)", ('Test Künstler',))

Erfolgreich: 1 Zeile(n) betroffen.


1

In [23]:
# Namen des Künstlers ändern
execute_db("""
    UPDATE artists 
    SET Name = ? 
    WHERE Name = ?
""", ('Neuer Name', 'Test Künstler'))

Erfolgreich: 1 Zeile(n) betroffen.


1

In [24]:
# Überprüfen, ob die Änderung erfolgreich war
query_db("SELECT * FROM artists WHERE Name LIKE 'Neuer%'")

1 Ergebnisse gefunden.
------------------------------------------------------------
(277, 'Neuer Name')


[(277, 'Neuer Name')]

### Beispiel 16: Mehrere Spalten gleichzeitig aktualisieren

In [28]:
# Vorher: Kundeninformationen anzeigen
query_db("SELECT FirstName, LastName, City, Country FROM customers WHERE CustomerId = 1")

1 Ergebnisse gefunden.
------------------------------------------------------------
('Luís', 'Gonçalves', 'São José dos Campos', 'Brazil')


[('Luís', 'Gonçalves', 'São José dos Campos', 'Brazil')]

In [29]:
# Stadt und Land eines Kunden ändern (Beispiel - wird wieder rückgängig gemacht)
execute_db("""
    UPDATE customers 
    SET City = 'München', Country = 'Germany'
    WHERE CustomerId = 1
""")

Erfolgreich: 1 Zeile(n) betroffen.


1

In [30]:
# Nachher: Änderung überprüfen
query_db("SELECT FirstName, LastName, City, Country FROM customers WHERE CustomerId = 1")

1 Ergebnisse gefunden.
------------------------------------------------------------
('Luís', 'Gonçalves', 'München', 'Germany')


[('Luís', 'Gonçalves', 'München', 'Germany')]

In [31]:
# Änderung rückgängig machen
execute_db("""
    UPDATE customers 
    SET City = 'São José dos Campos', Country = 'Brazil'
    WHERE CustomerId = 1
""")

Erfolgreich: 1 Zeile(n) betroffen.


1

### Übung 4.1: UPDATE anwenden

**Aufgabe:** Erstellen Sie einen neuen Künstler mit dem Namen "Übungskünstler" und ändern Sie den Namen dann zu "Mein Lieblingskünstler".

**Hinweise:**
- Nutzen Sie `execute_db()` für beide Operationen
- Verwenden Sie parametrisierte Abfragen

In [32]:
# Ihr Code hier


**Musterlösung:**

<details>
<summary>Lösung anzeigen</summary>

```python
# Schritt 1: Künstler erstellen
execute_db("INSERT INTO artists (Name) VALUES (?)", ('Übungskünstler',))

# Schritt 2: Namen ändern
execute_db("""
    UPDATE artists 
    SET Name = ? 
    WHERE Name = ?
""", ('Mein Lieblingskünstler', 'Übungskünstler'))

# Überprüfen
query_db("SELECT * FROM artists WHERE Name LIKE '%Lieblingskünstler%'")
```

**Erklärung:** Zuerst wird der Künstler mit INSERT erstellt, dann mit UPDATE umbenannt. Die WHERE-Klausel stellt sicher, dass nur der gewünschte Datensatz geändert wird.
</details>

---

### DELETE: Daten löschen

Der `DELETE`-Befehl entfernt Datensätze aus einer Tabelle.

**Syntax:**
```sql
DELETE FROM tabelle
WHERE bedingung;
```

**⚠️ KRITISCH:** Ohne `WHERE`-Klausel werden ALLE Zeilen gelöscht! Dies ist nicht rückgängig zu machen.

### Beispiel 17: Datensatz löschen

In [33]:
# Den zuvor erstellten Test-Künstler löschen
execute_db("DELETE FROM artists WHERE Name = ?", ('Neuer Name',))

Erfolgreich: 1 Zeile(n) betroffen.


1

In [34]:
# Überprüfen, ob die Löschung erfolgreich war
query_db("SELECT * FROM artists WHERE Name LIKE 'Neuer%'")

0 Ergebnisse gefunden.
------------------------------------------------------------


[]

### Best Practices für UPDATE und DELETE

1. **Immer WHERE verwenden**: Ohne WHERE werden ALLE Datensätze betroffen
2. **Erst SELECT, dann UPDATE/DELETE**: Testen Sie die WHERE-Bedingung zuerst mit SELECT
3. **Parametrisierte Abfragen**: Schützen vor SQL-Injection
4. **Backups erstellen**: Vor größeren Änderungen

In [35]:
# Gute Praxis: Erst prüfen, was gelöscht würde
query_db("SELECT * FROM artists WHERE Name LIKE '%Lieblingskünstler%'")

0 Ergebnisse gefunden.
------------------------------------------------------------


[]

In [36]:
# Dann löschen
execute_db("DELETE FROM artists WHERE Name LIKE '%Lieblingskünstler%'")

Erfolgreich: 0 Zeile(n) betroffen.


0

---

## Kapitel 5: Aggregatfunktionen und GROUP BY

### Einführung und Motivation

Bisher haben Sie einzelne Datensätze abgefragt. Oft benötigen Sie jedoch zusammenfassende Informationen: Wie viele Tracks gibt es? Was ist der durchschnittliche Preis? Welches Genre hat die meisten Songs?

**Aggregatfunktionen** berechnen einen einzelnen Wert aus einer Menge von Zeilen. In Kombination mit **GROUP BY** können Sie diese Berechnungen für Gruppen von Daten durchführen – ähnlich wie die groupby()-Funktion in Pandas (Notebook 17).

### Die wichtigsten Aggregatfunktionen

| Funktion | Beschreibung | Beispiel |
|----------|--------------|----------|
| `COUNT(*)` | Zählt alle Zeilen | Anzahl der Tracks |
| `COUNT(spalte)` | Zählt Nicht-NULL-Werte | Anzahl der Tracks mit Komponist |
| `SUM(spalte)` | Summiert Werte | Gesamtumsatz |
| `AVG(spalte)` | Berechnet Durchschnitt | Durchschnittspreis |
| `MIN(spalte)` | Findet Minimum | Günstigster Preis |
| `MAX(spalte)` | Findet Maximum | Teuerster Preis |

### Beispiel 18: Einfache Aggregatfunktionen

In [25]:
# Anzahl aller Tracks
query_db("SELECT COUNT(*) FROM tracks")

1 Ergebnisse gefunden.
------------------------------------------------------------
(3503,)


[(3503,)]

In [38]:
# Durchschnittspreis aller Tracks
query_db("SELECT AVG(UnitPrice) FROM tracks")

1 Ergebnisse gefunden.
------------------------------------------------------------
(1.0508050242649156,)


[(1.0508050242649156,)]

In [39]:
# Minimaler und maximaler Preis
query_db("SELECT MIN(UnitPrice), MAX(UnitPrice) FROM tracks")

1 Ergebnisse gefunden.
------------------------------------------------------------
(0.99, 1.99)


[(0.99, 1.99)]

### Beispiel 19: COUNT mit und ohne NULL

In [40]:
# COUNT(*) zählt alle Zeilen
query_db("SELECT COUNT(*) FROM tracks")

1 Ergebnisse gefunden.
------------------------------------------------------------
(3503,)


[(3503,)]

In [41]:
# COUNT(spalte) zählt nur Nicht-NULL-Werte
query_db("SELECT COUNT(Composer) FROM tracks")

1 Ergebnisse gefunden.
------------------------------------------------------------
(2525,)


[(2525,)]

Es gibt 3503 Tracks, aber nur 2525 haben einen Komponisten eingetragen.

---

### GROUP BY: Gruppierte Auswertungen

Mit `GROUP BY` können Aggregatfunktionen für Gruppen von Daten berechnet werden.

**Syntax:**
```sql
SELECT spalte, AGGREGAT(spalte)
FROM tabelle
GROUP BY spalte;
```

### Beispiel 20: Tracks pro Genre zählen

In [42]:
# Anzahl der Tracks pro Genre
query_db("""
    SELECT genres.Name, COUNT(*) as Anzahl
    FROM tracks
    INNER JOIN genres ON tracks.GenreId = genres.GenreId
    GROUP BY genres.Name
    ORDER BY Anzahl DESC
""")

25 Ergebnisse gefunden.
------------------------------------------------------------
('Rock', 1297)
('Latin', 579)
('Metal', 374)
('Alternative & Punk', 332)
('Jazz', 130)
('TV Shows', 93)
('Blues', 81)
('Classical', 74)
('Drama', 64)
('R&B/Soul', 61)
... und 15 weitere Ergebnisse


[('Rock', 1297),
 ('Latin', 579),
 ('Metal', 374),
 ('Alternative & Punk', 332),
 ('Jazz', 130),
 ('TV Shows', 93),
 ('Blues', 81),
 ('Classical', 74),
 ('Drama', 64),
 ('R&B/Soul', 61),
 ('Reggae', 58),
 ('Pop', 48),
 ('Soundtrack', 43),
 ('Alternative', 40),
 ('Hip Hop/Rap', 35),
 ('Electronica/Dance', 30),
 ('World', 28),
 ('Heavy Metal', 28),
 ('Sci Fi & Fantasy', 26),
 ('Easy Listening', 24),
 ('Comedy', 17),
 ('Bossa Nova', 15),
 ('Science Fiction', 13),
 ('Rock And Roll', 12),
 ('Opera', 1)]

**Erklärung:** Die Abfrage zählt Tracks pro Genre. `GROUP BY genres.Name` gruppiert die Daten, `COUNT(*)` zählt die Einträge pro Gruppe. `ORDER BY Anzahl DESC` sortiert nach Häufigkeit.

### Beispiel 21: Durchschnittswerte pro Gruppe

In [43]:
# Durchschnittliche Tracklänge pro Genre (in Minuten)
query_db("""
    SELECT genres.Name, ROUND(AVG(tracks.Milliseconds) / 60000, 2) as DurchschnittMin
    FROM tracks
    INNER JOIN genres ON tracks.GenreId = genres.GenreId
    GROUP BY genres.Name
    ORDER BY DurchschnittMin DESC
""")

25 Ergebnisse gefunden.
------------------------------------------------------------
('Sci Fi & Fantasy', 48.53)
('Science Fiction', 43.76)
('Drama', 42.92)
('TV Shows', 35.75)
('Comedy', 26.42)
('Metal', 5.16)
('Electronica/Dance', 5.05)
('Heavy Metal', 4.96)
('Classical', 4.9)
('Jazz', 4.86)
... und 15 weitere Ergebnisse


[('Sci Fi & Fantasy', 48.53),
 ('Science Fiction', 43.76),
 ('Drama', 42.92),
 ('TV Shows', 35.75),
 ('Comedy', 26.42),
 ('Metal', 5.16),
 ('Electronica/Dance', 5.05),
 ('Heavy Metal', 4.96),
 ('Classical', 4.9),
 ('Jazz', 4.86),
 ('Rock', 4.73),
 ('Blues', 4.51),
 ('Alternative', 4.4),
 ('Reggae', 4.12),
 ('Soundtrack', 4.07),
 ('Alternative & Punk', 3.91),
 ('Latin', 3.88),
 ('Pop', 3.82),
 ('World', 3.75),
 ('R&B/Soul', 3.67),
 ('Bossa Nova', 3.66),
 ('Easy Listening', 3.15),
 ('Hip Hop/Rap', 2.97),
 ('Opera', 2.91),
 ('Rock And Roll', 2.24)]

### Übung 5.1: Aggregatfunktionen anwenden

**Aufgabe:** Berechnen Sie den Gesamtumsatz (SUM) pro Kunde. Zeigen Sie Vorname, Nachname und Gesamtsumme an, sortiert nach Umsatz absteigend.

**Hinweise:**
- Verknüpfen Sie `customers` und `invoices` über `CustomerId`
- Die Rechnungssumme steht in der Spalte `Total`
- Gruppieren Sie nach Kunde

In [44]:
# Ihr Code hier


**Musterlösung:**

<details>
<summary>Lösung anzeigen</summary>

```python
# Gesamtumsatz pro Kunde
query_db("""
    SELECT customers.FirstName, customers.LastName, SUM(invoices.Total) as Gesamtumsatz
    FROM customers
    INNER JOIN invoices ON customers.CustomerId = invoices.CustomerId
    GROUP BY customers.CustomerId
    ORDER BY Gesamtumsatz DESC
""")
```

**Erklärung:** Der JOIN verbindet Kunden mit ihren Rechnungen. GROUP BY fasst alle Rechnungen eines Kunden zusammen. SUM() addiert die Total-Werte dieser Rechnungen.
</details>

---

### HAVING: Filter für Gruppen

Die `WHERE`-Klausel filtert einzelne Zeilen **vor** der Gruppierung. Um Gruppen nach der Aggregation zu filtern, verwenden Sie `HAVING`.

**Syntax:**
```sql
SELECT spalte, AGGREGAT(spalte)
FROM tabelle
GROUP BY spalte
HAVING bedingung_auf_aggregat;
```

### Beispiel 22: Nur Genres mit mehr als 100 Tracks

In [45]:
# Genres mit mehr als 100 Tracks
query_db("""
    SELECT genres.Name, COUNT(*) as Anzahl
    FROM tracks
    INNER JOIN genres ON tracks.GenreId = genres.GenreId
    GROUP BY genres.Name
    HAVING Anzahl > 100
    ORDER BY Anzahl DESC
""")

5 Ergebnisse gefunden.
------------------------------------------------------------
('Rock', 1297)
('Latin', 579)
('Metal', 374)
('Alternative & Punk', 332)
('Jazz', 130)


[('Rock', 1297),
 ('Latin', 579),
 ('Metal', 374),
 ('Alternative & Punk', 332),
 ('Jazz', 130)]

### Übung 5.2: HAVING anwenden

**Aufgabe:** Finden Sie alle Kunden, die mehr als 40€ Gesamtumsatz haben. Zeigen Sie Name und Gesamtumsatz an.

**Hinweise:**
- Basieren Sie auf der vorherigen Übung
- Fügen Sie eine HAVING-Klausel hinzu

In [46]:
# Ihr Code hier


**Musterlösung:**

<details>
<summary>Lösung anzeigen</summary>

```python
# Kunden mit mehr als 40€ Umsatz
query_db("""
    SELECT customers.FirstName, customers.LastName, SUM(invoices.Total) as Gesamtumsatz
    FROM customers
    INNER JOIN invoices ON customers.CustomerId = invoices.CustomerId
    GROUP BY customers.CustomerId
    HAVING Gesamtumsatz > 40
    ORDER BY Gesamtumsatz DESC
""")
```

**Erklärung:** HAVING filtert die Gruppen nach der Aggregation. Nur Kunden, deren SUM(Total) größer als 40 ist, werden angezeigt.
</details>

---

## Abschlussübungen

Die folgenden Aufgaben testen Ihr Verständnis der in diesem Notebook erlernten Konzepte. Bearbeiten Sie die Aufgaben selbstständig und vergleichen Sie Ihre Lösung anschließend mit den Musterlösungen am Ende des Notebooks.

### Teil 1: Grundlegende Anwendung

**Kompetenzstufe**: Anwenden

Diese Aufgaben testen die direkte Anwendung der erlernten Konzepte.

**Aufgabe 1:** Schreiben Sie eine Abfrage, die alle Kunden aus den USA anzeigt, deren Vorname mit "J" beginnt. Sortieren Sie die Ergebnisse alphabetisch nach Nachname.

In [47]:
# Arbeitsbereich für Aufgabe 1


**Aufgabe 2:** Zeigen Sie alle Alben an, die von Künstlern stammen, deren Name "The" enthält. Nutzen Sie einen INNER JOIN zwischen `albums` und `artists`. Geben Sie Albumtitel und Künstlername aus.

In [48]:
# Arbeitsbereich für Aufgabe 2


### Teil 2: Transfer und Problemlösung

**Kompetenzstufe**: Analysieren

Diese Aufgaben erfordern die Kombination mehrerer Konzepte und eigenständiges Problemlösen.

**Aufgabe 3:** Erstellen Sie eine Umsatzanalyse nach Land. Zeigen Sie für jedes Land die Anzahl der Kunden und den durchschnittlichen Umsatz pro Kunde an. Filtern Sie nur Länder mit mindestens 2 Kunden. Sortieren Sie nach durchschnittlichem Umsatz absteigend.

*Tipp: Sie benötigen einen JOIN, GROUP BY und HAVING.*

In [49]:
# Arbeitsbereich für Aufgabe 3


**Aufgabe 4:** Finden Sie die 5 längsten Tracks pro Genre. Zeigen Sie Genrename, Trackname und Länge in Minuten an. Sortieren Sie innerhalb jedes Genres nach Länge absteigend.

*Tipp: Diese Aufgabe erfordert einen Subquery oder eine kreative Kombination von Konzepten.*

In [50]:
# Arbeitsbereich für Aufgabe 4


---

## Zusammenfassung

In diesem Notebook haben Sie fortgeschrittene SQL-Techniken für SQLite kennengelernt:

| Konzept | SQL-Syntax | Anwendungsfall |
|---------|------------|----------------|
| Filtern | `WHERE bedingung` | Gezielt Datensätze auswählen |
| Logische Operatoren | `AND`, `OR`, `NOT` | Komplexe Bedingungen formulieren |
| Spezialoperatoren | `BETWEEN`, `IN`, `LIKE` | Bereiche, Listen, Muster abfragen |
| Sortieren | `ORDER BY spalte [ASC\|DESC]` | Ergebnisse ordnen |
| INNER JOIN | `JOIN ... ON ...` | Tabellen über Schlüssel verbinden |
| LEFT JOIN | `LEFT JOIN ... ON ...` | Alle Zeilen der linken Tabelle |
| UPDATE | `UPDATE ... SET ... WHERE` | Daten ändern |
| DELETE | `DELETE FROM ... WHERE` | Daten löschen |
| Aggregatfunktionen | `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` | Zusammenfassungen berechnen |
| Gruppieren | `GROUP BY spalte` | Aggregatfunktionen pro Gruppe |
| Gruppenfilter | `HAVING bedingung` | Gruppen nach Aggregation filtern |

**Zentrale Erkenntnisse**:
- WHERE filtert vor der Gruppierung, HAVING danach
- JOINs verbinden Tabellen und ermöglichen komplexe Abfragen über mehrere Tabellen
- UPDATE und DELETE sind destruktiv – immer WHERE verwenden!
- Aggregatfunktionen ermöglichen statistische Auswertungen direkt in SQL

**Abschluss der Datenbankmodule**: Mit den Notebooks 20-23 haben Sie einen umfassenden Überblick über relationale Datenbanken und deren praktische Anwendung mit Python und SQLite erhalten.

---

## Musterlösungen

<details>
<summary>Lösung zu Aufgabe 1</summary>

```python
# Kunden aus USA mit Vorname beginnend mit J
query_db("""
    SELECT FirstName, LastName, City 
    FROM customers 
    WHERE Country = 'USA' AND FirstName LIKE 'J%'
    ORDER BY LastName ASC
""")
```

**Erklärung**:
- `Country = 'USA'` filtert nach US-Kunden
- `FirstName LIKE 'J%'` findet Vornamen, die mit J beginnen
- `AND` kombiniert beide Bedingungen
- `ORDER BY LastName ASC` sortiert alphabetisch

**Häufige Fehler**:
- LIKE 'J' statt LIKE 'J%' (findet nur genau 'J')
- Vergessen der Anführungszeichen bei Textwerten
</details>

<details>
<summary>Lösung zu Aufgabe 2</summary>

```python
# Alben von Künstlern mit "The" im Namen
query_db("""
    SELECT albums.Title, artists.Name
    FROM albums
    INNER JOIN artists ON albums.ArtistId = artists.ArtistId
    WHERE artists.Name LIKE '%The%'
    ORDER BY artists.Name
""")
```

**Erklärung**:
- INNER JOIN verbindet Alben mit Künstlern
- LIKE '%The%' findet "The" an beliebiger Stelle im Namen
- Die Sortierung nach Künstlername gruppiert Alben desselben Künstlers
</details>

<details>
<summary>Lösung zu Aufgabe 3</summary>

```python
# Umsatzanalyse nach Land
query_db("""
    SELECT 
        customers.Country,
        COUNT(DISTINCT customers.CustomerId) as AnzahlKunden,
        ROUND(AVG(invoices.Total), 2) as DurchschnittUmsatz
    FROM customers
    INNER JOIN invoices ON customers.CustomerId = invoices.CustomerId
    GROUP BY customers.Country
    HAVING AnzahlKunden >= 2
    ORDER BY DurchschnittUmsatz DESC
""")
```

**Erklärung**:
- JOIN verbindet Kunden mit ihren Rechnungen
- COUNT(DISTINCT) zählt eindeutige Kunden pro Land
- AVG berechnet den Durchschnitt der Rechnungen
- HAVING filtert Länder mit weniger als 2 Kunden
- ROUND formatiert auf 2 Dezimalstellen

**Alternative Ansätze**:
- SUM(Total)/COUNT(DISTINCT CustomerId) statt AVG für exakten Durchschnitt pro Kunde
</details>

<details>
<summary>Lösung zu Aufgabe 4</summary>

```python
# Die längsten Tracks pro Genre (vereinfachte Lösung: Top 10 insgesamt)
query_db("""
    SELECT 
        genres.Name as Genre,
        tracks.Name as Track,
        ROUND(tracks.Milliseconds / 60000.0, 2) as Minuten
    FROM tracks
    INNER JOIN genres ON tracks.GenreId = genres.GenreId
    ORDER BY genres.Name, tracks.Milliseconds DESC
""", limit=20)
```

**Erklärung**:
- Diese vereinfachte Lösung sortiert nach Genre und dann nach Länge
- Die Division durch 60000.0 konvertiert Millisekunden in Minuten
- Für exakt die Top 5 pro Genre wäre ein Subquery mit ROW_NUMBER() nötig, was fortgeschrittenes SQL ist

**Fortgeschrittene Lösung mit Subquery** (zur Inspiration):
```python
# Exakt Top 5 pro Genre (komplexer)
query_db("""
    SELECT Genre, Track, Minuten
    FROM (
        SELECT 
            genres.Name as Genre,
            tracks.Name as Track,
            ROUND(tracks.Milliseconds / 60000.0, 2) as Minuten,
            ROW_NUMBER() OVER (PARTITION BY genres.GenreId ORDER BY tracks.Milliseconds DESC) as rn
        FROM tracks
        INNER JOIN genres ON tracks.GenreId = genres.GenreId
    )
    WHERE rn <= 5
    ORDER BY Genre, Minuten DESC
""", limit=50)
```
</details>

---

## Aufräumen

Zum Abschluss entfernen wir eventuelle Testdaten:

In [51]:
# Testdaten aufräumen
execute_db("DELETE FROM artists WHERE Name LIKE 'Test%' OR Name LIKE 'Neuer%' OR Name LIKE '%Lieblingskünstler%' OR Name LIKE 'Übungs%'")
print("Aufräumen abgeschlossen.")

Erfolgreich: 0 Zeile(n) betroffen.
Aufräumen abgeschlossen.
